In [1]:
import pandas as pd
from pathlib import Path
import re
from typing import List, Tuple, cast

# Configuración de rutas
proyecto_root = Path("/home/agusriscos/proyectos/ARPTools")
carpeta_grafos = proyecto_root / "data" / "16dic25" / "grafos"

print(f"Buscando archivos de metadata en: {carpeta_grafos}")


Buscando archivos de metadata en: /home/agusriscos/proyectos/ARPTools/data/16dic25/grafos


In [2]:
def extraer_imf_y_transformacion(ruta_archivo: Path) -> Tuple[str, str]:
    """
    Extrae el IMF y la transformación de la ruta del archivo de metadata.
    
    Parameters
    ----------
    ruta_archivo : Path
        Ruta completa al archivo de metadata.
        
    Returns
    -------
    Tuple[str, str]
        Tupla con (imf, transformacion) extraídos de la ruta.
        
    Examples
    --------
    >>> ruta = Path("data/16dic25/grafos/nvg/imf_1/grafo_nvg_imf_1_metadata.csv")
    >>> imf, transformacion = extraer_imf_y_transformacion(ruta)
    >>> print(imf, transformacion)
    imf_1 nvg
    """
    # La estructura es: grafos/{transformacion}/{imf}/grafo_{transformacion}_{imf}_metadata.csv
    partes = ruta_archivo.parts
    # Buscar el índice de 'grafos' en la ruta
    try:
        idx_grafos = partes.index('grafos')
        transformacion = partes[idx_grafos + 1]
        imf = partes[idx_grafos + 2]
        return imf, transformacion
    except (ValueError, IndexError):
        # Si no encontramos la estructura esperada, intentar extraer del nombre del archivo
        nombre_archivo = ruta_archivo.name
        # Patrón: grafo_{transformacion}_{imf}_metadata.csv
        match = re.match(r'grafo_(\w+)_(.+?)_metadata\.csv', nombre_archivo)
        if match:
            return match.group(2), match.group(1)
        raise ValueError(f"No se pudo extraer IMF y transformación de: {ruta_archivo}")


def leer_todos_metadata(carpeta_base: Path) -> pd.DataFrame:
    """
    Lee todos los archivos de metadata y los combina en un único DataFrame.
    
    Agrega dos columnas nuevas: 'imf' y 'transformacion' indicando a qué
    IMF y transformación pertenece cada registro de metadata.
    
    Parameters
    ----------
    carpeta_base : Path
        Ruta base donde se encuentran los archivos de metadata.
        Estructura esperada: {carpeta_base}/{transformacion}/{imf}/grafo_{transformacion}_{imf}_metadata.csv
        
    Returns
    -------
    pd.DataFrame
        DataFrame con todos los metadata combinados, incluyendo las columnas
        'imf' y 'transformacion'.
        
    Examples
    --------
    >>> carpeta = Path("data/16dic25/grafos")
    >>> df_completo = leer_todos_metadata(carpeta)
    >>> print(df_completo.head())
    """
    archivos_metadata = list(carpeta_base.rglob("*_metadata.csv"))
    
    if not archivos_metadata:
        raise FileNotFoundError(
            f"No se encontraron archivos de metadata en: {carpeta_base}"
        )
    
    print(f"Encontrados {len(archivos_metadata)} archivos de metadata")
    
    lista_dataframes = []
    
    for archivo in archivos_metadata:
        try:
            # Leer el archivo CSV
            df_temp = pd.read_csv(archivo)
            
            # Extraer IMF y transformación de la ruta
            imf, transformacion = extraer_imf_y_transformacion(archivo)
            
            # Agregar columnas nuevas
            df_temp['imf'] = imf
            df_temp['transformacion'] = transformacion
            
            lista_dataframes.append(df_temp)
            
        except Exception as e:
            print(f"Error al procesar {archivo}: {e}")
            continue
    
    if not lista_dataframes:
        raise ValueError("No se pudo leer ningún archivo de metadata")
    
    # Combinar todos los DataFrames
    df_completo = cast(pd.DataFrame, pd.concat(lista_dataframes, ignore_index=True))
    
    # Reordenar columnas para que 'imf' y 'transformacion' estén al principio
    columnas = ['imf', 'transformacion'] + [col for col in df_completo.columns 
                                            if col not in ['imf', 'transformacion']]
    df_completo = df_completo[columnas]
    
    return df_completo  # type: ignore[return-value]


In [3]:
# Leer todos los archivos de metadata
df_metadata_completo = leer_todos_metadata(carpeta_grafos)

print(f"\nDataFrame completo creado:")
print(f"  - Total de registros: {len(df_metadata_completo)}")
print(f"  - Total de columnas: {len(df_metadata_completo.columns)}")
print(f"\nColumnas: {list(df_metadata_completo.columns)}")
print(f"\nPrimeras filas:")
df_metadata_completo.head()


Encontrados 33 archivos de metadata

DataFrame completo creado:
  - Total de registros: 33
  - Total de columnas: 12

Columnas: ['imf', 'transformacion', 'id_imf', 'num_nodes', 'num_edges', 'num_features', 'archivo_features', 'archivo_edges', 'tau', 'dim_embedding', 'algoritmo_distancia', 'umbral_recurrencia']

Primeras filas:


,imf,transformacion,id_imf,num_nodes,num_edges,num_features,archivo_features,archivo_edges,tau,dim_embedding,algoritmo_distancia,umbral_recurrencia
0,imf_7,hvg,IMF_7,3490,6757,1,grafo_hvg_imf_7_features.parquet,grafo_hvg_imf_7_edges.parquet,NaN,NaN,NaN,NaN
1,imf_8,hvg,IMF_8,3490,6170,1,grafo_hvg_imf_8_features.parquet,grafo_hvg_imf_8_edges.parquet,NaN,NaN,NaN,NaN
2,imf_2,hvg,IMF_2,3490,6962,1,grafo_hvg_imf_2_features.parquet,grafo_hvg_imf_2_edges.parquet,NaN,NaN,NaN,NaN
3,imf_9,hvg,IMF_9,3490,5293,1,grafo_hvg_imf_9_features.parquet,grafo_hvg_imf_9_edges.parquet,NaN,NaN,NaN,NaN
4,residuo,hvg,Residuo,3490,3489,1,grafo_hvg_residuo_features.parquet,grafo_hvg_residuo_edges.parquet,NaN,NaN,NaN,NaN


In [4]:
# Resumen por transformación e IMF
print("Resumen por transformación:")
print(df_metadata_completo.groupby('transformacion').size())
print("\nResumen por IMF:")
print(df_metadata_completo.groupby('imf').size())
print("\nResumen combinado (transformación x IMF):")
print(df_metadata_completo.groupby(['transformacion', 'imf']).size().unstack(fill_value=0))


Resumen por transformación:
transformacion
hvg            11
nvg            11
recurrencia    11
dtype: int64

Resumen por IMF:
imf
imf_1      3
imf_10     3
imf_2      3
imf_3      3
imf_4      3
imf_5      3
imf_6      3
imf_7      3
imf_8      3
imf_9      3
residuo    3
dtype: int64

Resumen combinado (transformación x IMF):
imf             imf_1  imf_10  imf_2  imf_3  imf_4  imf_5  imf_6  imf_7  \
transformacion                                                            
hvg                 1       1      1      1      1      1      1      1   
nvg                 1       1      1      1      1      1      1      1   
recurrencia         1       1      1      1      1      1      1      1   

imf             imf_8  imf_9  residuo  
transformacion                         
hvg                 1      1        1  
nvg                 1      1        1  
recurrencia         1      1        1  


In [ ]:
# Guardar el DataFrame completo
archivo_salida = proyecto_root / "data" / "16dic25" / "metadata_completo.csv"
df_metadata_completo.to_csv(archivo_salida, index=False)
print(f"✓ Tabla completa guardada en: {archivo_salida}")


✓ Tabla completa guardada en: /home/agusriscos/proyectos/ARPTools/data/16dic25/metadata_completo.csv
✓ Tabla completa guardada en formato parquet: /home/agusriscos/proyectos/ARPTools/data/16dic25/metadata_completo.parquet
